# Clase 5 · Regresión lineal y regularización

**Analítica de Datos** · Maestría en Ciencias del Comportamiento · Universidad de San Andrés

**Primavera 2026 · 05/09/2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomdamelio/analitica_de_datos_alumnos/blob/main/clases/clase-05/notebooks/clase05_r.ipynb)

---

Esta es la versión en R de la notebook de la clase. Cubre exactamente los mismos pasos, los mismos
datos y los mismos números que la de Python: si seguiste una, la otra no trae contenido nuevo.

La pregunta de hoy es **¿cuánto bienestar laboral podemos predecir a partir del salario?**, y de ahí
en adelante, **¿cuánto mejora si usamos más variables?**

> **El modelo que mejor ajusta los datos que ya viste no es el que mejor predice los que vienen.**

| # | Paso | Qué hacemos |
|---|---|---|
| 1 | Armar la tabla | unir `nimbus_clima` con `nimbus_salario` |
| 2 | Una recta a ojo | mover β₀ y β₁ a mano y mirar los residuos |
| 3 | Mínimos cuadrados | programar la fórmula, sin `lm()` |
| 4 | Medir el ajuste | RSE y R², a mano y después con `lm()` |
| 5 | Varias variables | ir agregando predictores y ver qué pasa |
| 6 | El número honesto | partir en entrenamiento y testeo, y volver a medir |

> **Ojo con los dos "bienestar".** El de hoy es `bienestar_laboral`, un índice de 0 a 100 de la
> encuesta de clima 2026. **No** es el `bienestar` diario del piloto de la fruta (escala 1 a 7).

## 1. Armar la tabla

`clima` tiene una fila por empleado. `salario` tiene una fila por empleado **y por año**, así que
primero hay que quedarse con un año.

In [ ]:
library(dplyr)
library(ggplot2)

# Todo sale del espejo público de la materia. Si cambia de lugar, se toca esta línea.
BASE <- "https://raw.githubusercontent.com/tomdamelio/analitica_de_datos_alumnos/main/data/toy-nimbus/"

clima <- read.csv(paste0(BASE, "nimbus_clima.csv"))
salario <- read.csv(paste0(BASE, "nimbus_salario.csv"))

cat("clima  ", dim(clima), "\n")
cat("salario", dim(salario), "\n")
head(clima, 3)

### ✏️ Consigna 1

Uní las dos tablas para tener, en una sola fila por empleado, su salario y su bienestar.

**a)** `salario` tiene tres años por persona. Quedate con **2025**.

**b)** Unilas por la columna que comparten. Las dos tienen `empleado_id`.

In [ ]:
salario_2025 <- salario |>
  # TODO: completá el año que nos interesa
  filter(anio == ___) |>
  select(empleado_id, salario_mensual)

datos <- clima |>
  # TODO: completá la columna por la que se unen las dos tablas
  inner_join(salario_2025, by = "___") |>
  mutate(salario = salario_mensual / 1e6)   # en millones, para leerlo cómodo

stopifnot(nrow(datos) == 600)
stopifnot(!any(duplicated(datos$empleado_id)))
cat(dim(datos), "\n")
head(datos[, c("empleado_id", "salario", "bienestar_laboral")], 3)


In [ ]:
correlacion <- cor(datos$salario, datos$bienestar_laboral)
stopifnot(correlacion > 0.5, correlacion < 0.65)
cat(sprintf("correlación salario / bienestar: %.3f\n", correlacion))
summary(datos[, c("salario", "bienestar_laboral")])

### El punto de partida: el scatter

In [ ]:
# Las dos variables sueltas: se usan en casi todas las celdas de acá en adelante.
x <- datos$salario
y <- datos$bienestar_laboral

nube <- ggplot(datos, aes(salario, bienestar_laboral)) +
  geom_point(alpha = 0.35, colour = "#33404a", size = 1.4) +
  labs(x = "Salario mensual (millones de $)", y = "Bienestar laboral (0-100)") +
  ylim(0, 100) +
  theme_minimal(base_size = 12)

nube + ggtitle("Nimbus: 600 empleados")

## 2. Una recta a ojo

Una recta son dos números: la ordenada al origen β₀ y la pendiente β₁.

$$\hat{y} = \beta_0 + \beta_1 x$$

donde
- $\hat{y}$ es el bienestar que la recta **predice**,
- $x$ es el salario,
- $\beta_0$ es dónde cruza el eje vertical,
- $\beta_1$ es cuánto sube el bienestar por cada millón más de salario.

En R no hay sliders como en Colab con Python, así que probamos tres rectas a la vez y comparamos.
El **RSS** es la suma de los residuos al cuadrado, y es lo que dice cuál es mejor:

$$\text{RSS} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### ✏️ Consigna 2

Programá el RSS: calculá el residuo de cada persona y sumá todos esos residuos **al cuadrado**.

In [ ]:
rss <- function(b0, b1) {
  # TODO: el residuo es el valor real menos el que predice la recta b0 + b1*x
  residuos <- y - (___ + ___ * x)
  # TODO: ¿a qué potencia hay que elevarlos antes de sumar?
  sum(residuos^___)
}

rss_mala <- rss(25, 26)
stopifnot(abs(rss_mala - 60773.88) < 0.1)
cat(sprintf("RSS de la recta (25,0 · 26,0): %s\n", format(round(rss_mala), big.mark = ".")))


In [ ]:
# Tres candidatas, para ver que el RSS ordena lo que el ojo intuye.
candidatas <- data.frame(
  nombre = c("muy plana", "razonable", "muy empinada"),
  b0 = c(45, 0, -70),
  b1 = c(10, 45, 100)
)
candidatas$rss <- mapply(rss, candidatas$b0, candidatas$b1)
candidatas$rss <- round(candidatas$rss)
candidatas

In [ ]:
nube +
  geom_abline(data = candidatas, aes(intercept = b0, slope = b1, colour = nombre), linewidth = 1.1) +
  scale_colour_manual(values = c("muy plana" = "#C8622A", "razonable" = "#00529B",
                                 "muy empinada" = "#1F7A4D"), name = NULL) +
  ggtitle("Tres rectas elegidas a ojo") +
  theme(legend.position = "bottom")

## 3. Mínimos cuadrados, sin `lm()`

Para este problema hay una fórmula exacta:

$$\hat{\beta}_1 = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}
\qquad
\hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

donde $\bar{x}$ y $\bar{y}$ son los promedios. La segunda dice algo lindo: la recta pasa
**siempre** por el punto de los dos promedios.

### ✏️ Consigna 3

Programá las dos fórmulas de arriba. Las dos usan los promedios `mx` y `my`, que ya están
calculados en la primera línea.

In [ ]:
minimos_cuadrados <- function(x, y) {
  mx <- mean(x); my <- mean(y)
  # TODO: arriba, cuánto se aparta cada uno de SU promedio; abajo, sólo el de x
  b1 <- sum((x - mx) * (y - ___)) / sum((x - ___)^2)
  # TODO: la recta pasa por (mx, my). Despejá b0 de my = b0 + b1 * mx
  b0 <- my - ___ * mx
  c(b0 = b0, b1 = b1)
}

ols <- minimos_cuadrados(x, y)
b0_ols <- unname(ols["b0"]); b1_ols <- unname(ols["b1"])

stopifnot(abs(b0_ols - (-31.0448)) < 0.01)
stopifnot(abs(b1_ols - 70.2997) < 0.01)
cat(sprintf("β₀ = %.2f\nβ₁ = %.2f\n", b0_ols, b1_ols))


In [ ]:
# Dos comprobaciones.
set.seed(42)
rss_ols <- rss(b0_ols, b1_ols)
al_azar <- replicate(4000, rss(b0_ols + rnorm(1, 0, 5), b1_ols + rnorm(1, 0, 5)))
stopifnot(rss_ols < min(al_azar))                       # nadie le gana
stopifnot(abs(b0_ols + b1_ols * mean(x) - mean(y)) < 1e-9)  # pasa por (x̄, ȳ)

cat(sprintf("RSS de la recta óptima:     %s\n", format(round(rss_ols), big.mark = ".")))
cat(sprintf("la mejor de 4.000 al azar:  %s\n", format(round(min(al_azar)), big.mark = ".")))
cat(sprintf("pasa por (x̄, ȳ) = (%.3f, %.2f): sí\n", mean(x), mean(y)))

In [ ]:
nube +
  geom_abline(intercept = 0, slope = 45, colour = "#C8622A", linewidth = 1.1, linetype = "dashed") +
  geom_abline(intercept = b0_ols, slope = b1_ols, colour = "#00529B", linewidth = 1.3) +
  ggtitle(sprintf("A ojo (RSS = %s) contra mínimos cuadrados (RSS = %s)",
                  format(round(rss(0, 45)), big.mark = "."),
                  format(round(rss_ols), big.mark = ".")))

## 4. Medir el ajuste: RSE y R²

$$\text{RSE} = \sqrt{\frac{\text{RSS}}{n - 2}}
\qquad
R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
\qquad
\text{TSS} = \sum_i (y_i - \bar{y})^2$$

donde el $-2$ del RSE es porque estimamos dos parámetros, y $R^2 = 0$ quiere decir que la recta no
le ganó a decir el promedio.

### ✏️ Consigna 4

Calculá las dos medidas. El TSS ya está hecho. Ojo con el denominador del RSE: no es `n`, es `n`
menos la cantidad de parámetros que estimamos.

In [ ]:
n <- length(y)
tss <- sum((y - mean(y))^2)

# TODO: ¿cuántos parámetros estimamos en una regresión simple?
rse <- sqrt(rss_ols / (n - ___))
# TODO: el R² compara el error de nuestra recta contra el del modelo del promedio
r2 <- 1 - ___ / tss

stopifnot(abs(rse - 9.2235) < 0.01)
stopifnot(abs(r2 - 0.3274) < 0.001)
cat(sprintf("TSS = %s\nRSE = %.2f puntos de bienestar\nR²  = %.3f\n",
            format(round(tss), big.mark = "."), rse, r2))


### Lo mismo con `lm()`

En R el ajuste es una línea. Lo importante es verificar que **da exactamente lo mismo**.

Un detalle de vocabulario: lo que `summary()` llama *Residual standard error* es nuestro RSE, y
*Multiple R-squared* es el R².

In [ ]:
modelo <- lm(bienestar_laboral ~ salario, data = datos)
resumen <- summary(modelo)

stopifnot(abs(coef(modelo)[[1]] - b0_ols) < 1e-9)
stopifnot(abs(coef(modelo)[[2]] - b1_ols) < 1e-9)
stopifnot(abs(resumen$r.squared - r2) < 1e-9)
stopifnot(abs(resumen$sigma - rse) < 1e-9)

cat(sprintf("lm()   β₀ = %.4f   β₁ = %.4f   R² = %.4f   RSE = %.4f\n",
            coef(modelo)[[1]], coef(modelo)[[2]], resumen$r.squared, resumen$sigma))
cat(sprintf("a mano β₀ = %.4f   β₁ = %.4f   R² = %.4f   RSE = %.4f\n",
            b0_ols, b1_ols, r2, rse))
resumen

En la salida de `summary()` fijate en la columna `Pr(>|t|)` del salario: es el p-valor del que
hablamos en la teoría. Un número minúsculo quiere decir que el cero no es una explicación razonable
de la pendiente que estamos viendo.

## 5. Varias variables

La encuesta tiene 19 predictores además del salario. Estos son los candidatos, cada uno solo, con
su R²:

In [ ]:
CANDIDATAS <- c("salario", "apoyo_equipo", "reconocimiento", "autonomia",
                "horas_extra_semana", "dias_home_office", "bono_anual_pct",
                "reuniones_semana", "mensajes_chat_dia", "puntualidad_pct",
                "horas_capacitacion", "distancia_oficina_km")

r2_de <- function(columnas, tabla = datos) {
  # R² del modelo que usa esas columnas. Es la función que vas a usar todo el rato.
  summary(lm(reformulate(columnas, response = "bienestar_laboral"), data = tabla))$r.squared
}

solas <- sapply(CANDIDATAS, function(c) r2_de(c))
round(sort(solas, decreasing = TRUE), 3)

Hay tres que explican algo y el resto está cerca de cero. La pregunta obvia: si junto las tres
buenas, ¿el R² es la suma de los tres?

### ✏️ Consigna 5

Compará las dos cosas: **sumar** los tres R² por separado contra ajustar **un solo modelo** con las
tres variables juntas.

Anotá tu predicción antes de correr la celda: ¿va a dar más, menos o igual?

In [ ]:
tres <- c("salario", "apoyo_equipo", "reconocimiento")

suma_individuales <- sum(solas[tres])
# TODO: ajustá UN modelo con las tres variables a la vez
juntas <- r2_de(___)

stopifnot(juntas < suma_individuales)
cat(sprintf("sumando los tres R2 por separado:   %.3f\n", suma_individuales))
cat(sprintf("el modelo con las tres juntas:      %.3f\n", juntas))
cat(sprintf("diferencia (información repetida):  %.3f\n", suma_individuales - juntas))


No se suman: la información se superpone.

### Un coeficiente no se lee solo

In [ ]:
solo_bono <- lm(bienestar_laboral ~ bono_anual_pct, data = datos)
con_salario <- lm(bienestar_laboral ~ bono_anual_pct + salario, data = datos)
r_bono_salario <- cor(datos$bono_anual_pct, datos$salario)

stopifnot(abs(coef(con_salario)[["bono_anual_pct"]]) < abs(coef(solo_bono)[["bono_anual_pct"]]) / 10)

cat(sprintf("beta del bono, solo:           %+.3f\n", coef(solo_bono)[["bono_anual_pct"]]))
cat(sprintf("beta del bono, con el salario: %+.3f\n", coef(con_salario)[["bono_anual_pct"]]))
cat(sprintf("correlación bono / salario:     %.3f\n", r_bono_salario))
cat(sprintf("R2 solo bono: %.3f   con las dos: %.3f\n",
            summary(solo_bono)$r.squared, summary(con_salario)$r.squared))

El bono parecía un predictor fuerte y resultó ser el salario escrito otra vez: en Nimbus el bono es
un porcentaje del sueldo.

### Ahora armá tu modelo

Esta es la parte en la que trabajás vos. Tenés `CANDIDATAS` y `r2_de()`.

In [ ]:
# Un ayudante para ir tanteando rápido.
probar <- function(columnas) {
  valor <- r2_de(columnas)
  cat(sprintf("R2 = %.3f   con %d variable(s): %s\n",
              valor, length(columnas), paste(columnas, collapse = ", ")))
  invisible(valor)
}

probar(c("salario"))
probar(c("salario", "apoyo_equipo"))
probar(c("salario", "apoyo_equipo", "reconocimiento"))
probar(c("salario", "apoyo_equipo", "reconocimiento", "puntualidad_pct"))
cat("\nVariables disponibles:\n")
cat(paste(CANDIDATAS, collapse = ", "), "\n")

Mirá las cuatro pruebas antes de seguir. Agregar `apoyo_equipo` y `reconocimiento` sube el R².
Agregar `puntualidad_pct`, que es ruido puro, lo sube **también**, aunque poquísimo. Guardate eso:
en un rato va a ser el centro de la clase.

### ✏️ Consigna 6

Armá tu propio modelo con **al menos tres variables** de `CANDIDATAS`, usando `probar()` para ir
tanteando. Cuando tengas una combinación que te convenza, ponela en `mis_variables`.

No busques la mejor de todas: buscá una que te parezca razonable.

In [ ]:
# TODO: poné acá las variables que elegiste. Al menos tres, todas de CANDIDATAS.
mis_variables <- c("___", "___", "___")

r2_mio <- r2_de(mis_variables)
probar(mis_variables)

stopifnot(length(mis_variables) >= 3)
stopifnot(all(mis_variables %in% CANDIDATAS))
stopifnot(r2_mio > 0.5)


## 6. El número honesto: entrenamiento y testeo

Todo lo que hiciste hasta acá lo mediste sobre las **mismas 600 personas** con las que armaste el
modelo. Es como corregir un examen con el machete a la vista.

### ✏️ Consigna 7

Partí la tabla dejando el **30%** para testeo, y evaluá **tu** modelo en los dos lados.

`set.seed(42)` es para que a todos les dé lo mismo y podamos comparar en clase.

In [ ]:
set.seed(42)
# TODO: qué proporción va a ENTRENAMIENTO (el resto queda para testeo)
indices <- sample(nrow(datos), size = round(___ * nrow(datos)))
entrenamiento <- datos[indices, ]
testeo <- datos[-indices, ]

r2_fuera <- function(modelo, evaluacion) {
  pred <- predict(modelo, newdata = evaluacion)
  1 - sum((evaluacion$bienestar_laboral - pred)^2) /
    sum((evaluacion$bienestar_laboral - mean(entrenamiento$bienestar_laboral))^2)
}

mio <- lm(reformulate(mis_variables, response = "bienestar_laboral"), data = entrenamiento)
r2_train_mio <- summary(mio)$r.squared
# TODO: el número honesto se mide en el conjunto que el modelo NO vio
r2_test_mio <- r2_fuera(mio, ___)

stopifnot(nrow(entrenamiento) == 420, nrow(testeo) == 180)
cat(sprintf("tu modelo: %s\n", paste(mis_variables, collapse = ", ")))
cat(sprintf("  R2 con TODA la data (lo de recién): %.3f\n", r2_mio))
cat(sprintf("  R2 en entrenamiento:                %.3f\n", r2_train_mio))
cat(sprintf("  R2 en testeo:                       %.3f\n", r2_test_mio))
cat(sprintf("  hueco entrenamiento - testeo:       %+.3f\n", r2_train_mio - r2_test_mio))


### ¿Se podía elegir mejor?

Comparemos tres modelos sobre la misma partición: sólo el salario, el tuyo, y uno con **todas** las
variables. Si agregar variables siempre mejorara, el último tendría que ganar.

In [ ]:
TODAS <- setdiff(names(datos), c("empleado_id", "bienestar_laboral", "salario_mensual"))

evaluar <- function(columnas) {
  m <- lm(reformulate(columnas, response = "bienestar_laboral"), data = entrenamiento)
  c(train = summary(m)$r.squared, test = r2_fuera(m, testeo))
}

filas <- list(
  list(nombre = "sólo el salario", cols = c("salario")),
  list(nombre = "el tuyo", cols = mis_variables),
  list(nombre = "TODAS", cols = TODAS)
)
comparacion <- do.call(rbind, lapply(filas, function(f) {
  r <- evaluar(f$cols)
  data.frame(modelo = f$nombre, variables = length(f$cols),
             train = r[["train"]], test = r[["test"]], hueco = r[["train"]] - r[["test"]])
}))

stopifnot(comparacion$train[3] > comparacion$train[2])
stopifnot(comparacion$hueco[3] > comparacion$hueco[2])

round_df <- comparacion
round_df[, 3:5] <- round(round_df[, 3:5], 3)
round_df

Ahí está la clase entera en una tabla. Con **todas** las variables el R² de entrenamiento es el más
alto de los tres, y el de testeo **no**. El hueco entre las dos columnas es el sobreajuste.

### La curva completa

In [ ]:
agregar_de_a_una <- function(columnas) {
  # Forward selection: en cada paso agrega la que más sube el R² de entrenamiento.
  elegidas <- character(0)
  restantes <- columnas
  historia <- list()
  while (length(restantes) > 0) {
    r2s <- sapply(restantes, function(c) evaluar(c(elegidas, c))[["train"]])
    mejor <- restantes[which.max(r2s)]
    elegidas <- c(elegidas, mejor)
    restantes <- setdiff(restantes, mejor)
    r <- evaluar(elegidas)
    historia[[length(historia) + 1]] <- data.frame(
      n_variables = length(elegidas), agregada = mejor,
      r2_entrenamiento = r[["train"]], r2_testeo = r[["test"]]
    )
  }
  do.call(rbind, historia)
}

historia <- agregar_de_a_una(TODAS)

# El R² de entrenamiento NUNCA puede bajar al agregar una variable: es matemático.
stopifnot(all(diff(historia$r2_entrenamiento) >= -1e-9))

mejor <- historia[which.max(historia$r2_testeo), ]
stopifnot(mejor$n_variables < length(TODAS))

cat(sprintf("mejor en testeo: %d variables, R2 = %.3f\n", mejor$n_variables, mejor$r2_testeo))
cat(sprintf("tu modelo:       %d variables, R2 = %.3f\n", length(mis_variables), r2_test_mio))
historia

In [ ]:
largo <- rbind(
  data.frame(n = historia$n_variables, r2 = historia$r2_entrenamiento, conjunto = "entrenamiento"),
  data.frame(n = historia$n_variables, r2 = historia$r2_testeo, conjunto = "testeo")
)

ggplot(largo, aes(n, r2, colour = conjunto)) +
  geom_line(linewidth = 1) +
  geom_point(size = 1.8) +
  geom_vline(xintercept = mejor$n_variables, linetype = "dashed", colour = "#33404a") +
  annotate("point", x = length(mis_variables), y = r2_test_mio,
           shape = 8, size = 4, colour = "#1F7A4D", stroke = 1.2) +
  annotate("text", x = length(mis_variables) + 0.4, y = r2_test_mio - 0.05,
           label = "tu modelo", size = 3.2, colour = "#1F7A4D", hjust = 0) +
  annotate("text", x = mejor$n_variables + 0.4, y = mejor$r2_testeo + 0.05,
           label = sprintf("mejor en testeo: %d variables", mejor$n_variables),
           size = 3.2, colour = "#33404a", hjust = 0) +
  scale_colour_manual(values = c(entrenamiento = "#1F6FB4", testeo = "#C8622A"), name = NULL) +
  labs(x = "Variables en el modelo", y = "R2",
       title = "El entrenamiento nunca baja. El testeo sí.") +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

La curva azul sube siempre; la naranja sube, hace un máximo y se cae. Ese máximo es la cantidad de
variables que conviene usar, y **no es la mayor**. La estrella verde es dónde quedó tu modelo.

### Lo que queda: penalizar en vez de descartar

`glmnet` ajusta ridge (`alpha = 0`) y lasso (`alpha = 1`). Estandariza por defecto, que es lo que
hay que hacer siempre antes de penalizar.

> **Los números no coinciden dígito a dígito con los de Python, y los lambda tampoco.** `glmnet`
> parametriza la penalización de otra manera que `scikit-learn`: acá lambda = 10 para ridge y
> lambda = 1 para lasso hacen el mismo trabajo que alpha = 10 y alpha = 0,5 allá. Lo que sí coincide
> es el orden: con pocos datos mínimos cuadrados se desarma y las dos penalizadas aguantan.

### ✏️ Consigna 8

Comprobá con los datos lo que dijimos en la teoría. Entrená los tres modelos con **sólo 30
personas** y las 19 variables, y comparalos sobre el resto.

En `glmnet`, `alpha = 0` es ridge y `alpha = 1` es lasso. No confundas ese `alpha` con el `alpha`
de Python, que ahí es el lambda.

In [ ]:
library(glmnet)

set.seed(11)
# TODO: cuántas personas usamos para entrenar
chicos <- sample(nrow(datos), size = ___)
chico <- datos[chicos, ]
resto <- datos[-chicos, ]

X_chico <- as.matrix(chico[, TODAS]); X_resto <- as.matrix(resto[, TODAS])
y_chico <- chico$bienestar_laboral;   y_resto <- resto$bienestar_laboral

r2_resto <- function(pred) 1 - sum((y_resto - pred)^2) / sum((y_resto - mean(y_chico))^2)

ols_chico <- lm(reformulate(TODAS, response = "bienestar_laboral"), data = chico)
# TODO: alpha = 0 es ridge, alpha = 1 es lasso
ridge <- glmnet(X_chico, y_chico, alpha = ___, lambda = 10)
lasso <- glmnet(X_chico, y_chico, alpha = ___, lambda = 1)

resultados <- c(
  "mínimos cuadrados" = r2_resto(predict(ols_chico, newdata = resto)),
  "Ridge"             = r2_resto(as.numeric(predict(ridge, newx = X_resto))),
  "Lasso"             = r2_resto(as.numeric(predict(lasso, newx = X_resto)))
)

stopifnot(resultados[["mínimos cuadrados"]] < 0)
stopifnot(resultados[["Ridge"]] > 0, resultados[["Lasso"]] > 0)
for (nombre in names(resultados)) {
  cat(sprintf("%18s: R2 de testeo = %+.3f\n", nombre, resultados[[nombre]]))
}


Con 30 personas y 19 variables, mínimos cuadrados da un R² **negativo**: predice peor que decir el
promedio y no pensar.

## Hoja de referencia

| Qué | Fórmula | En R |
|---|---|---|
| Residuo | $e_i = y_i - \hat{y}_i$ | `residuals(modelo)` |
| RSS | $\sum_i e_i^2$ | `sum(residuals(modelo)^2)` |
| RSE | $\sqrt{\text{RSS}/(n-2)}$ | `summary(modelo)$sigma` |
| R² | $1 - \text{RSS}/\text{TSS}$ | `summary(modelo)$r.squared` |
| Ajustar | | `lm(y ~ x1 + x2, data = datos)` |
| Predecir | | `predict(modelo, newdata = testeo)` |
| Ridge | RSS $+ \lambda \sum \beta_j^2$ | `glmnet(X, y, alpha = 0)` |
| Lasso | RSS $+ \lambda \sum \lvert\beta_j\rvert$ | `glmnet(X, y, alpha = 1)` |

> **Cuidado con `var()` y `sd()` en R:** dividen por $n-1$. En numpy el default es $n$. Acá no
> molesta porque todo se calculó con sumas explícitas, pero si comparás resultados entre los dos
> lenguajes es la primera cosa a revisar.

## Para el trabajo práctico

Sobre el dataset de tu grupo:

1. Elegí una variable respuesta **numérica**.
2. Ajustá la regresión simple con el predictor más prometedor y reportá el **R² de testeo**.
3. Agregá predictores de a uno y armá la curva de R² de entrenamiento contra R² de testeo, como la
   figura de la sección 6. Marcá dónde se separan.
4. Una línea de conclusión: ¿el mejor modelo es el que más variables tiene? ¿Por qué?